# PE6201 A2 — Group 3 · Problem A
Outline-aligned Colab runner. Core logic lives in `src/`; this notebook is only setup, smoke tests, controlled experiments and demo.


## 0. Clone / pull the full repository


In [ ]:
import os
if not os.path.exists('/content/PE6201-A2-Group3'):
    !git clone https://github.com/Tonya0719/PE6201-A2-Group3.git
%cd /content/PE6201-A2-Group3
!pip -q install -r requirements.txt


## 1. Configuration and imports
Default backend is scripted. For a live run, change only the controlled variables in `src/config.py` or below.


In [ ]:
from src import config
from src.agent_core import run_agent
from src.evaluation import run_evaluation, summarize_results
from src.tools import reset_outbox, PREAUTH_REQUIRED_CODES
print('backend:', config.BACKEND)
print('model:', config.MODEL)
print('preauth-required codes:', PREAUTH_REQUIRED_CODES)


## 2. Scripted smoke test — expected 4-turn full path
`CLM-8842`: claim → policy → batched coverage + preauth → Final/write.


In [ ]:
reset_outbox()
r = run_agent('CLM-8842', approved_for_write=True)
print('decision:', r.get('decision'))
print('turns:', r.get('turns'))
print('tools by turn:', [[c['name'] for c in h.get('tool_calls', [])] for h in r.get('tool_history', [])])


## 3. Variable path smoke tests
Expected shape: duplicate/injection ≈2 turns; policy-level escalation ≈3; full path ≈4.


In [ ]:
for cid in ['CLM-8933','CLM-8910','CLM-8842','CLM-8888']:
    reset_outbox()
    x = run_agent(cid, approved_for_write=True)
    print(cid, x.get('decision'), x.get('trigger'), 'turns=', x.get('turns'))


## 4. D4 / D5(a) scripted evaluation


In [ ]:
records = run_evaluation()
summarize_results(records)


## 5. Live debugging
The live prompt enforces exactly one JSON object per model turn and forbids model-generated observations. Use `debug_raw=True` to inspect the exact model response.


In [ ]:
# config.BACKEND = 'live'
# config.MODEL = 'google/gemini-2.5-flash-lite'
# reset_outbox()
# live = run_agent('CLM-8842', approved_for_write=False, debug_raw=True)
# live


## 6. D2(b) V1 → V2 controlled comparison
Same cheap live model, same cases, same loop/guardrails; only tool descriptor version changes.


In [ ]:
# config.BACKEND = 'live'
# v1 = run_evaluation(tool_spec_version='v1')
# v2 = run_evaluation(tool_spec_version='v2')
# print('V1', summarize_results(v1))
# print('V2', summarize_results(v2))


## 7. D2(c) sequential vs batched/parallel comparison
The outline-optimized path batches coverage ×N + preauthorisation ×M in Turn 3. Compare turns/tokens/cost/correctness rather than assuming it is better.


In [ ]:
# seq = run_evaluation(parallel_enabled=False)
# par = run_evaluation(parallel_enabled=True)
# print('Sequential', summarize_results(seq))
# print('Parallel', summarize_results(par))


## 8. D5(b), D6, D7
Use the same committed V2 prompt/eval/loop for each live model; consume saved measured results in D6; reproduce D7 failures as working Agent − X using scripted backend.


## 9. Git from Colab
Work on your own branch, not `main`.


In [ ]:
# BRANCH = 'feature/your-branch-name'
# COMMIT_MESSAGE = 'Describe your update'
# !git checkout main
# !git pull origin main
# !git checkout {BRANCH} || git checkout -b {BRANCH}
# !git add .
# !git commit -m "{COMMIT_MESSAGE}"
# !git push -u origin {BRANCH}
